In [1]:
# =========================================================
# Notebook 06 — Final Evaluation and Dissertation Figures
# =========================================================
#
# Purpose:
# Evaluate the trained regression and classification models
# on the out-of-time test set and generate the quantitative
# tables and figures used in the dissertation.
#
# Evaluates:
# - Regression early-warning pipeline:
#   - Naive persistence
#   - OLS (Delta)
#   - XGBoost (Delta)
# - Classifiers used in the dashboard:
#   - risk_xgb_classifier
#   - overperf_xgb_classifier
#
# Generates:
# - evaluation tables (CSV)
# - confusion matrices
# - threshold sensitivity outputs
# - residual/error summaries
# - SHAP summary and bar plots
# - optional SHAP dependence plots
# - ROC and precision-recall curves
# - calibration curve for the underperformance classifier
# - probability distribution plots with governance thresholds
#
# Inputs:
# - school_panel_final.parquet
# - delta_xgb_model.joblib
# - delta_model_features.joblib
# - delta_ols_model.joblib
# - risk_xgb_classifier.joblib
# - risk_model_features.joblib
# - overperf_xgb_classifier.joblib
# - overperf_model_features.joblib
#
# Stored in:
# - data/processed/
#
# Outputs:
# - figures saved to:
#   - data/processed/figures_eval/
# - evaluation CSV outputs saved to:
#   - data/processed/
#
# Role in the pipeline:
# This notebook is the final evaluation stage of the project.
# It operationalises the dissertation’s model comparison,
# threshold analysis, explainability outputs, and dashboard
# classifier validation using the held-out 2022–2023 data.
#
# Reproducibility note:
# The file paths in this notebook are currently configured for
# the author’s local machine. These should be updated if the
# project is run in a different environment.
#
# Important:
# This notebook was developed on a local Windows environment.
# Users reproducing the pipeline should replace absolute paths
# with environment-specific paths or relative project paths.
#
# Methodological note:
# This notebook follows the dissertation’s out-of-time split:
# - train years: 2017–2019
# - test years: 2022–2023
#
# The key policy threshold used for underperformance is:
# - Progress 8 < -0.5
#
# Governance note:
# Probability outputs are also examined using dashboard-style
# RAG thresholds:
# - Green < 0.40
# - Amber < 0.60
# - Red >= 0.60
# =========================================================

# === ONE-CELL: FULL PROJECT EVALUATION + DISSERTATION FIGURES ===
# Evaluates:
#   - Regression early-warning pipeline: Naive vs OLS(Delta) vs XGBoost(Delta)
#   - Classifiers used in dashboard: risk_xgb_classifier, overperf_xgb_classifier
# Generates figures:
#   - Table CSVs for dissertation
#   - MAE/threshold metrics + robustness sweep
#   - Confusion matrices
#   - Error distribution summaries
#   - SHAP summary + bar plots (+ optional dependence plots) for XGB models
#   - ROC + PR curves for classifiers
#   - Probability distribution plots with governance thresholds (0.40/0.60)

import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap

from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    mean_absolute_error,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
    roc_curve,
    precision_recall_curve,
)

# =========================================================
# Configuration
# =========================================================
#
# This section defines:
# - input artefact locations
# - figure output directory
# - evaluation years
# - policy thresholds
# - dashboard RAG thresholds
# - SHAP sampling controls
#
# If reproducing this notebook on another machine, update
# these paths before running the notebook.
# =========================================================

# -----------------------------
# PATHS
# -----------------------------
BASE_DIR = r"C:\Users\kiero\Documents\msc-dissertation\data"
PROCESSED = os.path.join(BASE_DIR, "processed")
FIG_DIR = os.path.join(PROCESSED, "figures_eval")
os.makedirs(FIG_DIR, exist_ok=True)

PANEL_PATH = os.path.join(PROCESSED, "school_panel_final.parquet")

# Regression (delta) artefacts
DELTA_XGB_PATH = os.path.join(PROCESSED, "delta_xgb_model.joblib")
DELTA_FEATURES_PATH = os.path.join(PROCESSED, "delta_model_features.joblib")
DELTA_OLS_PATH = os.path.join(PROCESSED, "delta_ols_model.joblib")

# Classifier artefacts used by dashboard
RISK_XGB_PATH = os.path.join(PROCESSED, "risk_xgb_classifier.joblib")
RISK_FEATS_PATH = os.path.join(PROCESSED, "risk_model_features.joblib")
OVER_XGB_PATH = os.path.join(PROCESSED, "overperf_xgb_classifier.joblib")
OVER_FEATS_PATH = os.path.join(PROCESSED, "overperf_model_features.joblib")

assert os.path.exists(PANEL_PATH), f"Missing panel file: {PANEL_PATH}"

# -----------------------------
# CONFIG (align with dissertation/tool)
# -----------------------------
TRAIN_YEARS = [2017, 2018, 2019]
TEST_YEARS  = [2022, 2023]

# Policy threshold used in dissertation (floor standard)
P8_UNDER_THR = -0.5

# "Overperformance" definition is project-dependent.
# If your classifier was trained on a different threshold, change this to match your training.
P8_OVER_THR = 0.0  # common choice: P8 > 0

# Dashboard banding thresholds (as per app.py)
GREEN_MAX = 0.40
AMBER_MAX = 0.60

# SHAP sampling (keeps it fast + readable)
SHAP_N = 2000
DEPENDENCE_TOPK = 2  # keep minimal for dissertation clarity

# =========================================================
# Helper Functions
# =========================================================
#
# These helper functions support:
# - figure saving
# - classification metrics
# - confusion matrix plotting
# - ROC / PR plotting
# - SHAP value handling
# - numeric coercion of feature matrices
#
# They are used to keep the evaluation logic clearer and more
# auditable.
# =========================================================

# -----------------------------
# HELPERS
# -----------------------------
def savefig(name: str):
    path = os.path.join(FIG_DIR, name)
    plt.tight_layout()
    plt.savefig(path, dpi=200, bbox_inches="tight")
    plt.close()
    return path

def cls_metrics(y_true_bin: np.ndarray, y_pred_bin: np.ndarray) -> dict:
    tn, fp, fn, tp = confusion_matrix(y_true_bin, y_pred_bin, labels=[0, 1]).ravel()
    return {
        "Precision": precision_score(y_true_bin, y_pred_bin, zero_division=0),
        "Recall": recall_score(y_true_bin, y_pred_bin, zero_division=0),
        "F1": f1_score(y_true_bin, y_pred_bin, zero_division=0),
        "TP": int(tp), "FP": int(fp), "TN": int(tn), "FN": int(fn)
    }

def plot_confusion(y_true, y_pred, title, fname):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    plt.figure()
    plt.imshow(cm, interpolation="nearest")
    plt.title(title)
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.xticks([0, 1], ["0", "1"])
    plt.yticks([0, 1], ["0", "1"])
    for (i, j), v in np.ndenumerate(cm):
        plt.text(j, i, str(v), ha="center", va="center")
    return savefig(fname)

def plot_roc_pr(y_true, y_score, title_prefix, fname_prefix):
    # ROC
    fpr, tpr, _ = roc_curve(y_true, y_score)
    auc = roc_auc_score(y_true, y_score) if len(np.unique(y_true)) == 2 else np.nan
    plt.figure()
    plt.plot(fpr, tpr)
    plt.title(f"{title_prefix} ROC (AUC={auc:.3f})")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    roc_path = savefig(f"{fname_prefix}_roc.png")

    # PR
    prec, rec, _ = precision_recall_curve(y_true, y_score)
    ap = average_precision_score(y_true, y_score) if len(np.unique(y_true)) == 2 else np.nan
    plt.figure()
    plt.plot(rec, prec)
    plt.title(f"{title_prefix} Precision-Recall (AP={ap:.3f})")
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    pr_path = savefig(f"{fname_prefix}_pr.png")

    return roc_path, pr_path, float(auc), float(ap)

def safe_shap_values(explainer, X: pd.DataFrame):
    sv = explainer.shap_values(X)
    # binary classifier often returns list [class0, class1]
    if isinstance(sv, list) and len(sv) >= 2:
        return sv[1]
    # some return 3D arrays
    arr = np.array(sv)
    if arr.ndim == 3:
        return arr[:, :, 1]
    return sv

def ensure_numeric_df(X: pd.DataFrame) -> pd.DataFrame:
    return X.apply(pd.to_numeric, errors="coerce")

# =========================================================
# Step 1 — Load Final Panel and Build Evaluation Dataset
# =========================================================
#
# This step loads the final modelling panel, ensures required
# columns are present, constructs the delta target, and applies
# the dissertation’s out-of-time split.
# =========================================================

# -----------------------------
# LOAD DATA
# -----------------------------
df = pd.read_parquet(PANEL_PATH).copy()
df["YEAR_START"] = pd.to_numeric(df["YEAR_START"], errors="coerce")

for col in ["TARGET_P8", "TARGET_P8_LAG1"]:
    if col not in df.columns:
        raise KeyError(f"Expected column missing: {col}")

df["TARGET_P8"] = pd.to_numeric(df["TARGET_P8"], errors="coerce")
df["TARGET_P8_LAG1"] = pd.to_numeric(df["TARGET_P8_LAG1"], errors="coerce")
df["TARGET_P8_DELTA"] = df["TARGET_P8"] - df["TARGET_P8_LAG1"]

# Keep rows where delta + year defined (used for delta framework)
df_delta = df.dropna(subset=["YEAR_START", "TARGET_P8", "TARGET_P8_LAG1", "TARGET_P8_DELTA"]).copy()

train_df = df_delta[df_delta["YEAR_START"].isin(TRAIN_YEARS)].copy()
test_df  = df_delta[df_delta["YEAR_START"].isin(TEST_YEARS)].copy()

print("Train rows:", train_df.shape, "| Test rows:", test_df.shape)

# Common mask for full-P8 evaluation (lag + actual exist)
y_test_p8 = test_df["TARGET_P8"].astype(float).values
y_test_p8_lag = test_df["TARGET_P8_LAG1"].astype(float).values
mask = np.isfinite(y_test_p8) & np.isfinite(y_test_p8_lag)

# =========================================================
# Step 2 — Evaluate Regression Early-Warning Pipeline
# =========================================================
#
# Models compared:
# - Naive persistence baseline
# - OLS (Delta)
# - XGBoost (Delta)
#
# Regression predictions are reconstructed into full P8 by:
#   predicted P8 = lagged P8 + predicted delta
#
# Outputs:
# - MAE table
# - thresholded early-warning metrics at P8 < -0.5
# - confusion matrices
# - scatter plot
# - threshold sensitivity sweep
# =========================================================

# -----------------------------
# 1) REGRESSION EARLY-WARNING PIPELINE (Naive vs OLS vs XGB)
# -----------------------------
assert os.path.exists(DELTA_XGB_PATH), f"Missing: {DELTA_XGB_PATH}"
assert os.path.exists(DELTA_FEATURES_PATH), f"Missing: {DELTA_FEATURES_PATH}"
assert os.path.exists(DELTA_OLS_PATH), f"Missing: {DELTA_OLS_PATH}"

delta_xgb = joblib.load(DELTA_XGB_PATH)
delta_ols = joblib.load(DELTA_OLS_PATH)
delta_feats = joblib.load(DELTA_FEATURES_PATH)

X_train = ensure_numeric_df(train_df.reindex(columns=delta_feats))
y_train_delta = train_df["TARGET_P8_DELTA"].astype(float).values

X_test = ensure_numeric_df(test_df.reindex(columns=delta_feats))

# Naive persistence (P8_pred = lag1)
p8_pred_naive = y_test_p8_lag

# XGB delta
delta_pred_xgb = delta_xgb.predict(X_test)
p8_pred_xgb = y_test_p8_lag + delta_pred_xgb

# OLS delta baseline (median impute; drop all-NaN cols in TRAIN)
all_nan_cols = X_train.columns[X_train.isna().all()].tolist()
X_train_ols_df = X_train.drop(columns=all_nan_cols) if all_nan_cols else X_train
X_test_ols_df  = X_test.drop(columns=all_nan_cols) if all_nan_cols else X_test

imputer = SimpleImputer(strategy="median")
X_train_ols = imputer.fit_transform(X_train_ols_df)   # fit on train only
X_test_ols  = imputer.transform(X_test_ols_df)

delta_pred_ols = delta_ols.predict(X_test_ols)
p8_pred_ols = y_test_p8_lag + delta_pred_ols

# Continuous metrics
mae_naive = mean_absolute_error(y_test_p8[mask], p8_pred_naive[mask])
mae_ols   = mean_absolute_error(y_test_p8[mask], p8_pred_ols[mask])
mae_xgb   = mean_absolute_error(y_test_p8[mask], p8_pred_xgb[mask])

# Threshold metrics @ -0.5
y_true_bin = (y_test_p8[mask] < P8_UNDER_THR).astype(int)
base_rate = float(y_true_bin.mean())

y_pred_naive = (p8_pred_naive[mask] < P8_UNDER_THR).astype(int)
y_pred_ols   = (p8_pred_ols[mask] < P8_UNDER_THR).astype(int)
y_pred_xgb   = (p8_pred_xgb[mask] < P8_UNDER_THR).astype(int)

m_naive = cls_metrics(y_true_bin, y_pred_naive)
m_ols   = cls_metrics(y_true_bin, y_pred_ols)
m_xgb   = cls_metrics(y_true_bin, y_pred_xgb)

reg_table = pd.DataFrame([
    {"Model": "Naive Persistence", "MAE": mae_naive, **m_naive},
    {"Model": "OLS (Delta)",       "MAE": mae_ols,   **m_ols},
    {"Model": "XGBoost (Delta)",   "MAE": mae_xgb,   **m_xgb},
])

print("\n=== REGRESSION (Delta->P8) Out-of-time @ -0.5 ===")
print(f"Base rate (P8 < {P8_UNDER_THR}): {base_rate*100:.2f}%")
display(reg_table)

reg_csv = os.path.join(PROCESSED, "eval_regression_table.csv")
reg_table.to_csv(reg_csv, index=False)
print("Saved:", reg_csv)

# Confusion matrices (regression thresholded)
plot_confusion(y_true_bin, y_pred_naive, "Naive: Confusion Matrix @ -0.5", "cm_naive_reg.png")
plot_confusion(y_true_bin, y_pred_ols,   "OLS(Delta): Confusion Matrix @ -0.5", "cm_ols_reg.png")
plot_confusion(y_true_bin, y_pred_xgb,   "XGB(Delta): Confusion Matrix @ -0.5", "cm_xgb_reg.png")

# Predicted vs actual scatter (XGB + OLS + Naive)
plt.figure()
plt.scatter(y_test_p8[mask], p8_pred_naive[mask], s=6, alpha=0.4, label="Naive")
plt.scatter(y_test_p8[mask], p8_pred_ols[mask],   s=6, alpha=0.4, label="OLS(Delta)")
plt.scatter(y_test_p8[mask], p8_pred_xgb[mask],   s=6, alpha=0.4, label="XGB(Delta)")
plt.title("Actual vs Predicted Progress 8 (Out-of-time 2022–2023)")
plt.xlabel("Actual P8")
plt.ylabel("Predicted P8")
plt.legend()
savefig("scatter_actual_vs_pred.png")

# Error distribution summary (MedianAE + P90AE)
abs_err_naive = np.abs(y_test_p8[mask] - p8_pred_naive[mask])
abs_err_ols   = np.abs(y_test_p8[mask] - p8_pred_ols[mask])
abs_err_xgb   = np.abs(y_test_p8[mask] - p8_pred_xgb[mask])

err_table = pd.DataFrame([
    {"Model":"Naive", "MAE": abs_err_naive.mean(), "MedianAE": np.median(abs_err_naive), "P90AE": np.quantile(abs_err_naive, 0.90)},
    {"Model":"OLS",   "MAE": abs_err_ols.mean(),   "MedianAE": np.median(abs_err_ols),   "P90AE": np.quantile(abs_err_ols, 0.90)},
    {"Model":"XGB",   "MAE": abs_err_xgb.mean(),   "MedianAE": np.median(abs_err_xgb),   "P90AE": np.quantile(abs_err_xgb, 0.90)},
])
display(err_table)

err_csv = os.path.join(PROCESSED, "eval_error_distribution_summary.csv")
err_table.to_csv(err_csv, index=False)
print("Saved:", err_csv)

# Threshold sensitivity sweep (robustness)
thresholds = [-0.6, -0.55, -0.5, -0.45, -0.4]
rows = []
for t in thresholds:
    yt = (y_test_p8[mask] < t).astype(int)
    rows.append({"Threshold": t, "Model":"Naive",  **{k:v for k,v in cls_metrics(yt, (p8_pred_naive[mask] < t).astype(int)).items() if k in ["Precision","Recall","F1"]}})
    rows.append({"Threshold": t, "Model":"OLS",    **{k:v for k,v in cls_metrics(yt, (p8_pred_ols[mask]   < t).astype(int)).items() if k in ["Precision","Recall","F1"]}})
    rows.append({"Threshold": t, "Model":"XGBoost",**{k:v for k,v in cls_metrics(yt, (p8_pred_xgb[mask]   < t).astype(int)).items() if k in ["Precision","Recall","F1"]}})
sens = pd.DataFrame(rows)
display(sens)

sens_csv = os.path.join(PROCESSED, "eval_threshold_sensitivity.csv")
sens.to_csv(sens_csv, index=False)
print("Saved:", sens_csv)

# Plot threshold sensitivity (F1 vs threshold)
plt.figure()
for model in ["Naive", "OLS", "XGBoost"]:
    sub = sens[sens["Model"] == model]
    plt.plot(sub["Threshold"], sub["F1"], marker="o", label=model)
plt.title("Threshold Sensitivity: F1 vs Decision Threshold")
plt.xlabel("Threshold (P8 < threshold)")
plt.ylabel("F1")
plt.legend()
savefig("threshold_sensitivity_f1.png")

# =========================================================
# Step 3 — SHAP Analysis for Delta XGBoost Model
# =========================================================
#
# This step generates:
# - SHAP summary beeswarm plot
# - SHAP bar-importance plot
# - a small number of dependence plots for top features
#
# These support the dissertation’s explainability analysis.
# =========================================================

# -----------------------------
# SHAP for XGB Delta model (global + bar + tiny dependence set)
# -----------------------------
X_shap = X_test.sample(min(SHAP_N, len(X_test)), random_state=42)
expl = shap.TreeExplainer(delta_xgb)
sv = expl.shap_values(X_shap)

# Summary (beeswarm)
plt.figure()
shap.summary_plot(sv, X_shap, show=False)
plt.title("SHAP Summary – XGBoost (Delta model)")
savefig("shap_summary_delta_xgb.png")

# Bar importance
plt.figure()
shap.summary_plot(sv, X_shap, plot_type="bar", show=False, max_display=20)
plt.title("SHAP Importance (mean |SHAP|) – XGBoost (Delta model)")
savefig("shap_bar_delta_xgb.png")

# Optional: dependence plots for top-k drivers (kept minimal for dissertation)
imp = np.abs(sv).mean(axis=0)
top_idx = np.argsort(imp)[::-1][:DEPENDENCE_TOPK]
top_feats = [X_shap.columns[i] for i in top_idx]

for f in top_feats:
    plt.figure()
    shap.dependence_plot(f, sv, X_shap, show=False)
    plt.title(f"SHAP Dependence – {f}")
    savefig(f"shap_dependence_{f}.png")

# =========================================================
# Step 4 — Evaluate Dashboard Classifiers
# =========================================================
#
# This step evaluates the underperformance and overperformance
# classifiers used by the dashboard using:
# - confusion matrices
# - ROC curves
# - precision-recall curves
# - optional probability distributions with RAG thresholds
# =========================================================

# -----------------------------
# 2) CLASSIFIERS USED IN DASHBOARD (Underperformance + Overperformance)
# -----------------------------
def eval_classifier(model_path, feats_path, label_name, y_true_bin, title_prefix, fname_prefix,
                    prob_threshold=0.5, band_plot=False):
    """
    Evaluate a classifier using the held-out test set and
    generate core diagnostic plots.
    """
    model = joblib.load(model_path)
    feats = joblib.load(feats_path)

    Xtr = ensure_numeric_df(train_df.reindex(columns=feats))
    Xte = ensure_numeric_df(test_df.reindex(columns=feats))

    # Most XGB classifiers handle NaNs, but ensure numeric conversion
    # Predict probabilities
    proba = model.predict_proba(Xte)[:, 1]

    # Default classification threshold
    y_pred = (proba >= prob_threshold).astype(int)

    met = cls_metrics(y_true_bin, y_pred)
    auc = roc_auc_score(y_true_bin, proba) if len(np.unique(y_true_bin)) == 2 else np.nan
    ap  = average_precision_score(y_true_bin, proba) if len(np.unique(y_true_bin)) == 2 else np.nan

    # Confusion matrix
    plot_confusion(y_true_bin, y_pred, f"{title_prefix} Confusion Matrix (thr={prob_threshold})",
                   f"{fname_prefix}_cm.png")

    # ROC + PR
    roc_path, pr_path, _, _ = plot_roc_pr(y_true_bin, proba, title_prefix, fname_prefix)

    # Optional: probability distribution with RAG thresholds (dashboard-style)
    if band_plot:
        plt.figure()
        plt.hist(proba, bins=40)
        plt.axvline(GREEN_MAX)
        plt.axvline(AMBER_MAX)
        plt.title(f"{title_prefix} – Predicted probability distribution\n(Green<{GREEN_MAX}, Amber<{AMBER_MAX}, Red≥{AMBER_MAX})")
        plt.xlabel("Predicted probability")
        plt.ylabel("Count")
        savefig(f"{fname_prefix}_prob_dist_rag.png")

    return met, float(auc), float(ap)

# Underperformance classifier label (aligned to dissertation policy threshold)
y_under = (test_df["TARGET_P8"].astype(float).values < P8_UNDER_THR).astype(int)

if os.path.exists(RISK_XGB_PATH) and os.path.exists(RISK_FEATS_PATH):
    met_under, auc_under, ap_under = eval_classifier(
        RISK_XGB_PATH, RISK_FEATS_PATH,
        label_name="Underperformance",
        y_true_bin=y_under,
        title_prefix="Risk classifier (Underperformance)",
        fname_prefix="risk_classifier_under",
        prob_threshold=0.5,
        band_plot=True
    )
    print("\n=== CLASSIFIER: Underperformance ===")
    print("Precision/Recall/F1/TP/FP/TN/FN:", met_under)
    print("ROC AUC:", auc_under, "| PR AUC (AP):", ap_under)
else:
    print("\n[Skip] Underperformance classifier artefacts not found.")

# Overperformance classifier label (adjust threshold if your training used something else)
y_over = (test_df["TARGET_P8"].astype(float).values > P8_OVER_THR).astype(int)

if os.path.exists(OVER_XGB_PATH) and os.path.exists(OVER_FEATS_PATH):
    met_over, auc_over, ap_over = eval_classifier(
        OVER_XGB_PATH, OVER_FEATS_PATH,
        label_name="Overperformance",
        y_true_bin=y_over,
        title_prefix="Overperformance classifier",
        fname_prefix="overperf_classifier",
        prob_threshold=0.5,
        band_plot=True
    )
    print("\n=== CLASSIFIER: Overperformance ===")
    print("Precision/Recall/F1/TP/FP/TN/FN:", met_over)
    print("ROC AUC:", auc_over, "| PR AUC (AP):", ap_over)
else:
    print("\n[Skip] Overperformance classifier artefacts not found.")

# =========================================================
# Step 5 — SHAP Analysis for Classifier Models
# =========================================================
#
# If classifier artefacts are present, this step generates
# global SHAP summary and bar plots for each classifier.
# =========================================================

# -----------------------------
# SHAP for classifier models (global + bar) – if present
# -----------------------------
def shap_for_classifier(model_path, feats_path, title_prefix, fname_prefix):
    """
    Generate SHAP summary and bar plots for a classifier
    model on a sampled test subset.
    """
    model = joblib.load(model_path)
    feats = joblib.load(feats_path)
    Xte = ensure_numeric_df(test_df.reindex(columns=feats)).sample(min(SHAP_N, len(test_df)), random_state=42)

    expl = shap.TreeExplainer(model)
    sv = safe_shap_values(expl, Xte)

    plt.figure()
    shap.summary_plot(sv, Xte, show=False)
    plt.title(f"SHAP Summary – {title_prefix}")
    savefig(f"{fname_prefix}_shap_summary.png")

    plt.figure()
    shap.summary_plot(sv, Xte, plot_type="bar", show=False, max_display=20)
    plt.title(f"SHAP Importance – {title_prefix}")
    savefig(f"{fname_prefix}_shap_bar.png")

if os.path.exists(RISK_XGB_PATH) and os.path.exists(RISK_FEATS_PATH):
    shap_for_classifier(RISK_XGB_PATH, RISK_FEATS_PATH, "Risk classifier (Underperformance)", "risk_classifier")

if os.path.exists(OVER_XGB_PATH) and os.path.exists(OVER_FEATS_PATH):
    shap_for_classifier(OVER_XGB_PATH, OVER_FEATS_PATH, "Overperformance classifier", "overperf_classifier")

print("\n✅ Done. Figures saved to:", FIG_DIR)

# =========================================================
# Step 6 — Calibration Curve for Underperformance Classifier
# =========================================================
#
# This section evaluates probability calibration for the risk
# classifier, supporting the dissertation’s discussion of
# governance-oriented probability use and RAG classification.
# =========================================================

# ================================
# ADDITION 1: Calibration Curve (Risk Classifier)
# ================================
from sklearn.calibration import calibration_curve
from scipy.stats import wilcoxon
import numpy as np
import matplotlib.pyplot as plt

if os.path.exists(RISK_XGB_PATH) and os.path.exists(RISK_FEATS_PATH):

    risk_model = joblib.load(RISK_XGB_PATH)
    risk_feats = joblib.load(RISK_FEATS_PATH)

    X_test_risk = ensure_numeric_df(test_df.reindex(columns=risk_feats))
    y_under = (test_df["TARGET_P8"].astype(float).values < P8_UNDER_THR).astype(int)

    risk_probs = risk_model.predict_proba(X_test_risk)[:, 1]

    prob_true, prob_pred = calibration_curve(
        y_under, risk_probs, n_bins=10, strategy="quantile"
    )

    plt.figure()
    plt.plot(prob_pred, prob_true, marker="o")
    plt.plot([0, 1], [0, 1], linestyle="--")
    plt.xlabel("Mean predicted probability")
    plt.ylabel("Observed frequency")
    plt.title("Calibration Curve – Risk Classifier (Underperformance)")
    cal_path = savefig("risk_classifier_calibration.png")

    print("Saved:", cal_path)

else:
    print("Risk classifier not found — calibration skipped.")

# =========================================================
# Step 7 — Statistical Comparison of XGB vs OLS Errors
# =========================================================
#
# This section performs a Wilcoxon signed-rank test on paired
# absolute errors from the XGBoost and OLS delta regressors.
#
# This supports the dissertation’s discussion of whether
# observed error differences are statistically meaningful.
# =========================================================

# ================================
# ADDITION 2: Statistical Test (MAE comparison)
# ================================
from scipy.stats import wilcoxon

abs_err_xgb = np.abs(y_test_p8[mask] - p8_pred_xgb[mask])
abs_err_ols = np.abs(y_test_p8[mask] - p8_pred_ols[mask])

stat, p_value = wilcoxon(abs_err_xgb, abs_err_ols)

print("\n=== Statistical Comparison: XGB vs OLS Absolute Error ===")
print("Wilcoxon statistic:", stat)
print("p-value:", p_value)

if p_value < 0.05:
    print("Result: Statistically significant difference (p < 0.05)")
else:
    print("Result: Not statistically significant")

# =========================================================
# Step 8 — Residual Distribution Analysis
# =========================================================
#
# This section compares residual distributions across Naive,
# OLS, and XGBoost models and saves both a plot and a summary
# table.
# =========================================================

# ================================
# ADDITION 3: Residual Distributions
# ================================

resid_naive = y_test_p8[mask] - p8_pred_naive[mask]
resid_ols   = y_test_p8[mask] - p8_pred_ols[mask]
resid_xgb   = y_test_p8[mask] - p8_pred_xgb[mask]

plt.figure()
plt.hist(resid_naive, bins=50, alpha=0.5, label="Naive")
plt.hist(resid_ols, bins=50, alpha=0.5, label="OLS")
plt.hist(resid_xgb, bins=50, alpha=0.5, label="XGB")
plt.axvline(0)
plt.legend()
plt.title("Residual Distribution (Actual − Predicted P8)")
plt.xlabel("Residual")
plt.ylabel("Frequency")
resid_path = savefig("residual_distribution_comparison.png")

print("Saved:", resid_path)


# Summary statistics
resid_summary = pd.DataFrame([
    {"Model": "Naive", "Mean": resid_naive.mean(), "Std": resid_naive.std()},
    {"Model": "OLS",   "Mean": resid_ols.mean(),   "Std": resid_ols.std()},
    {"Model": "XGB",   "Mean": resid_xgb.mean(),   "Std": resid_xgb.std()},
])

print("\nResidual Summary:")
display(resid_summary)

resid_csv = os.path.join(PROCESSED, "eval_residual_summary.csv")
resid_summary.to_csv(resid_csv, index=False)
print("Saved:", resid_csv)

# =========================================================
# Outputs Produced
# =========================================================
#
# After successful execution, this notebook produces:
#
# Tables / CSV outputs:
# - eval_regression_table.csv
# - eval_error_distribution_summary.csv
# - eval_threshold_sensitivity.csv
# - eval_residual_summary.csv
#
# Figures:
# - confusion matrices
# - scatter plot
# - threshold sensitivity plot
# - SHAP summary / bar / dependence plots
# - ROC and precision-recall curves
# - probability distribution plots
# - calibration curve
# - residual distribution plot
#
# These outputs provide the quantitative and visual evidence
# used in the dissertation’s results and evaluation chapters.
# =========================================================

KeyboardInterrupt: 